# Code tạo Manifest csv


In [9]:
from pathlib import Path
import pandas as pd
import re
from collections import Counter


def get_last_number(text):
    """
    Lấy cụm số cuối chuỗi.
    Ví dụ:
    Subject7 -> 7
    Activity10 -> 10
    Trial2 -> 2
    Camera12 -> 12
    """
    match = re.search(r"\d+$", str(text))
    if match:
        return match.group()
    return None


def normalize_image_timestamp_strict(filename_stem):
    """
    Chỉ nhận timestamp ảnh đúng dạng:
    YYYY-MM-DDTHH_MM_SS
    hoặc
    YYYY-MM-DDTHH_MM_SS.microsecond

    Ví dụ hợp lệ:
    2018-07-06T12_43_05
    2018-07-06T12_43_05.389994

    Ví dụ bị bỏ qua:
    2018-07-06T12_43_05.389994_copy
    2018-07-06T12_43_05.389994_1
    2018-07-06T12_43_05.389994_2
    copy_2018-07-06T12_43_05.389994

    Output:
    2018-07-06T12:43:05.389994
    """

    text = str(filename_stem).strip()

    pattern = r"^\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2}(?:\.\d+)?$"

    if not re.match(pattern, text):
        return None

    date_part, time_part = text.split("T", 1)

    # Chỉ thay 2 dấu _ đầu trong phần HH_MM_SS thành :
    time_part = time_part.replace("_", ":", 2)

    return date_part + "T" + time_part


def build_frame_manifest(
    image_root,
    label_csv_path,
    output_csv_path,
    label_col="Label"
):
    image_root = Path(image_root)

    rows = []

    skipped_bad_structure = []
    skipped_bad_number = []
    skipped_non_timestamp_name = []

    suffix_counter = Counter()
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}

    total_files_seen = 0
    total_image_files_seen = 0

    for img_path in image_root.rglob("*"):

        if not img_path.is_file():
            continue

        total_files_seen += 1
        suffix_counter[img_path.suffix.lower()] += 1

        if img_path.suffix.lower() not in image_extensions:
            continue

        total_image_files_seen += 1

        parts = img_path.relative_to(image_root).parts

        # Expect:
        # Subject7 / Activity1 / Trial2 / Camera1 / timestamp.png
        if len(parts) != 5:
            skipped_bad_structure.append({
                "image_path": str(img_path),
                "relative_path": str(img_path.relative_to(image_root)),
                "parts": str(parts),
                "num_parts": len(parts)
            })
            continue

        subject_raw, activity_raw, trial_raw, camera_raw, filename = parts

        subject = get_last_number(subject_raw)
        activity = get_last_number(activity_raw)
        trial = get_last_number(trial_raw)
        camera = get_last_number(camera_raw)

        if None in [subject, activity, trial, camera]:
            skipped_bad_number.append({
                "image_path": str(img_path),
                "Subject_raw": subject_raw,
                "Activity_raw": activity_raw,
                "Trial_raw": trial_raw,
                "Camera_raw": camera_raw,
                "Subject": subject,
                "Activity": activity,
                "Trial": trial,
                "Camera": camera
            })
            continue

        timestamp = normalize_image_timestamp_strict(img_path.stem)

        if timestamp is None:
            skipped_non_timestamp_name.append({
                "image_path": str(img_path),
                "filename_stem": img_path.stem,
                "reason": "Tên file không phải timestamp gốc, có thể là copy/_1/_2 hoặc format khác"
            })
            continue

        rows.append({
            "image_path": str(img_path),
            "Subject": str(subject),
            "Activity": str(activity),
            "Trial": str(trial),
            "Camera": str(camera),
            "Timestamp": timestamp
        })

    image_df = pd.DataFrame(rows)

    label_df = pd.read_csv(label_csv_path)

    key_cols = ["Timestamp", "Subject", "Activity", "Trial"]

    missing_cols = [col for col in key_cols if col not in label_df.columns]
    if missing_cols:
        raise ValueError(f"CSV thiếu các cột bắt buộc: {missing_cols}")

    if label_col not in label_df.columns:
        print(f"Cảnh báo: Không thấy cột label `{label_col}` trong CSV.")

    # Chuẩn hóa key về string để merge
    for col in key_cols:
        image_df[col] = image_df[col].astype(str).str.strip()
        label_df[col] = label_df[col].astype(str).str.strip()

    # Không normalize phức tạp CSV.
    # CSV được giả định đang có Timestamp dạng dấu :
    # Ví dụ: 2018-07-06T12:43:05.389994

    # Kiểm tra duplicate ảnh trong cùng camera
    full_image_key_cols = ["Subject", "Activity", "Trial", "Camera", "Timestamp"]

    duplicate_image_same_camera = image_df[
        image_df.duplicated(subset=full_image_key_cols, keep=False)
    ]

    # Kiểm tra duplicate label trong CSV
    duplicate_label_keys = label_df[
        label_df.duplicated(subset=key_cols, keep=False)
    ]

    # Merge: CSV không có Camera, nên không merge theo Camera
    manifest_df = image_df.merge(
        label_df,
        on=key_cols,
        how="left",
        indicator=True
    )

    missing_labels_df = manifest_df[manifest_df["_merge"] == "left_only"]

    # Kiểm tra mỗi timestamp có bao nhiêu camera
    camera_count_df = (
        image_df
        .groupby(["Subject", "Activity", "Trial", "Timestamp"])["Camera"]
        .nunique()
        .reset_index(name="num_cameras")
    )

    timestamp_missing_camera_df = camera_count_df[
        camera_count_df["num_cameras"] < 2
    ]

    print("========== FILE EXTENSIONS ==========")
    print(suffix_counter)

    print("\n========== IMAGE SCAN ==========")
    print("Total files seen:", total_files_seen)
    print("Total image files seen:", total_image_files_seen)
    print("Valid timestamp image rows:", len(image_df))

    print("\n========== SKIPPED ==========")
    print("Skipped bad structure:", len(skipped_bad_structure))
    print("Skipped bad folder number:", len(skipped_bad_number))
    print("Skipped non-original timestamp filename:", len(skipped_non_timestamp_name))

    print("\n========== DUPLICATE CHECK ==========")
    print("Duplicate images same camera:", len(duplicate_image_same_camera))
    print("Duplicate label keys in CSV:", len(duplicate_label_keys))

    print("\n========== MERGE STATUS ==========")
    print(manifest_df["_merge"].value_counts())

    if label_col in manifest_df.columns:
        print("\n========== LABEL STATUS ==========")
        print("Images with label:", manifest_df[label_col].notna().sum())
        print("Images without label:", manifest_df[label_col].isna().sum())

    print("\n========== CAMERA COUNT PER TIMESTAMP ==========")
    print(camera_count_df["num_cameras"].value_counts().sort_index())
    print("Timestamp missing camera:", len(timestamp_missing_camera_df))

    print("\n========== NA COUNT BY COLUMN ==========")
    print(manifest_df.isna().sum())

    # Lưu debug files
    pd.DataFrame(skipped_bad_structure).to_csv(
        "skipped_bad_structure.csv",
        index=False
    )

    pd.DataFrame(skipped_bad_number).to_csv(
        "skipped_bad_number.csv",
        index=False
    )

    pd.DataFrame(skipped_non_timestamp_name).to_csv(
        "skipped_non_timestamp_name.csv",
        index=False
    )

    duplicate_image_same_camera.to_csv(
        "duplicate_images_same_camera.csv",
        index=False
    )

    duplicate_label_keys.to_csv(
        "duplicate_label_keys.csv",
        index=False
    )

    missing_labels_df.to_csv(
        "missing_labels.csv",
        index=False
    )

    camera_count_df.to_csv(
        "camera_count_per_timestamp.csv",
        index=False
    )

    timestamp_missing_camera_df.to_csv(
        "timestamp_missing_camera.csv",
        index=False
    )

    # Lưu manifest cuối cùng
    manifest_df.to_csv(output_csv_path, index=False)

    return manifest_df, image_df, label_df

In [11]:
manifest_df, image_df, label_df = build_frame_manifest(
    image_root="/kaggle/input/datasets/kitvanh/upfall789cam12",
    label_csv_path="/kaggle/input/datasets/kitvanh/sub789-labels/labels.csv",
    output_csv_path="/kaggle/working/frame_manifest.csv",
    label_col="Label"
)

========== FILE EXTENSIONS ==========
Counter({'.png': 100200, '.json': 20})

========== IMAGE SCAN ==========
Total files seen: 100220
Total image files seen: 100200
Valid timestamp image rows: 90086

========== SKIPPED ==========
Skipped bad structure: 0
Skipped bad folder number: 0
Skipped non-original timestamp filename: 10114

========== DUPLICATE CHECK ==========
Duplicate images same camera: 0
Duplicate label keys in CSV: 0

========== MERGE STATUS ==========
_merge
both          90086
left_only         0
right_only        0
Name: count, dtype: int64

========== LABEL STATUS ==========
Images with label: 90086
Images without label: 0

========== CAMERA COUNT PER TIMESTAMP ==========
num_cameras
1     8682
2    40702
Name: count, dtype: int64
Timestamp missing camera: 8682

========== NA COUNT BY COLUMN ==========
image_path     0
Subject        0
Activity       0
Trial          0
Camera         0
Timestamp      0
Tag            0
Label          0
Source_File    0
Row_Index      

In [12]:
manifest_df

,image_path,Subject,Activity,Trial,Camera,Timestamp,Tag,Label,Source_File,Row_Index,_merge
0,/kaggle/input/datasets/kitvanh/upfall789cam12/...,9,4,3,1,2018-07-09T12:06:28.223986,4,3,Subject9Activity4Trial3.csv,35,both
1,/kaggle/input/datasets/kitvanh/upfall789cam12/...,9,4,3,1,2018-07-09T12:06:30.740560,11,3,Subject9Activity4Trial3.csv,79,both
2,/kaggle/input/datasets/kitvanh/upfall789cam12/...,9,4,3,1,2018-07-09T12:06:32.491887,11,3,Subject9Activity4Trial3.csv,110,both
3,/kaggle/input/datasets/kitvanh/upfall789cam12/...,9,4,3,1,2018-07-09T12:06:30.269637,11,3,Subject9Activity4Trial3.csv,70,both
4,/kaggle/input/datasets/kitvanh/upfall789cam12/...,9,4,3,1,2018-07-09T12:06:35.445781,11,3,Subject9Activity4Trial3.csv,159,both
...,...,...,...,...,...,...,...,...,...,...,...
90081,/kaggle/input/datasets/kitvanh/upfall789cam12/...,8,7,1,2,2018-07-09T10:30:13.970569,7,6,Subject8Activity7Trial1.csv,886,both
90082,/kaggle/input/datasets/kitvanh/upfall789cam12/...,8,7,1,2,2018-07-09T10:29:56.336005,7,6,Subject8Activity7Trial1.csv,562,both
90083,/kaggle/input/datasets/kitvanh/upfall789cam12/...,8,7,1,2,2018-07-09T10:29:54.961341,7,6,Subject8Activity7Trial1.csv,538,both
90084,/kaggle/input/datasets/kitvanh/upfall789cam12/...,8,7,1,2,2018-07-09T10:30:14.297454,7,6,Subject8Activity7Trial1.csv,892,both


In [13]:
manifest_df.to_csv('frame_train_ready.csv')